# 🧠 Notebook 3: Deep Learning for Micro-Doppler Classification

Trains CNN (ResNet-style), Bidirectional LSTM, and Spectrogram Transformer
on micro-Doppler spectrograms. Includes Grad-CAM visualization.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    print(f"PyTorch version: {torch.__version__}")
    device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
except ImportError:
    print("PyTorch not installed. Run: pip install torch")
    raise

from Dataset.loader import load_dataset, get_iq_matrix, get_labels, get_train_test_split
from Preprocessing.spectrogram import compute_stft_spectrogram
from Deep_Learning.cnn import MicroDopplerCNN
from Deep_Learning.lstm import MicroDopplerLSTM, iq_to_tensor
from Deep_Learning.transformer import MicroDopplerTransformer
from Deep_Learning.dataset_torch import MicroDopplerDataset, get_dataloader, spectrogram_transform
from Deep_Learning.train_dl import train_model
from Evaluation.metrics import compute_metrics

plt.style.use('dark_background')
COLORS = ['#58a6ff', '#3fb950', '#f78166']
CLASS_NAMES = ['2-blade', '3-blade', '4-blade']

## 1. Load Data

In [ ]:
df = load_dataset('../helicopter_microdoppler_dataset.csv', nrows=5000)
X_iq_train, X_iq_test, y_train, y_test = get_train_test_split(df, test_size=0.2)

# Datasets with spectrogram transform
ds_train = MicroDopplerDataset(X_iq_train, y_train, transform=spectrogram_transform)
ds_test  = MicroDopplerDataset(X_iq_test,  y_test,  transform=spectrogram_transform)
dl_train = get_dataloader(ds_train, batch_size=64, shuffle=True)
dl_test  = get_dataloader(ds_test,  batch_size=64, shuffle=False)

# Inspect one batch
X_b, y_b = next(iter(dl_train))
print(f"Batch shape: {X_b.shape}  |  Labels: {set(y_b.numpy().tolist())}")
print(f"Train batches: {len(dl_train)}, Test batches: {len(dl_test)}")

## 2. Model Architectures

In [ ]:
models_info = {
    'CNN (ResNet-style)': MicroDopplerCNN(n_classes=3),
    'Transformer (ViT)':  MicroDopplerTransformer(n_classes=3, img_h=33, img_w=15),
}

for name, model in models_info.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {name:<25}  {n_params:>8,} parameters")

## 3. Train CNN

In [ ]:
cnn = MicroDopplerCNN(n_classes=3)
print("Training CNN...")
history_cnn = train_model(
    cnn, dl_train, dl_test,
    n_epochs=30, lr=1e-3, patience=7,
    model_name='cnn_nb3', device=device
)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Training Curves', fontsize=14, fontweight='bold')

epochs = range(1, len(history_cnn['train_loss']) + 1)

axes[0].plot(epochs, history_cnn['train_loss'], color=COLORS[0], linewidth=2, label='Train')
axes[0].plot(epochs, history_cnn['val_loss'],   color=COLORS[1], linewidth=2, linestyle='--', label='Validation')
axes[0].set_title('Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_cnn['val_acc'], color=COLORS[2], linewidth=2.5)
axes[1].fill_between(epochs, history_cnn['val_acc'], alpha=0.2, color=COLORS[2])
axes[1].set_title('Validation Accuracy', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].axhline(y=max(history_cnn['val_acc']), color='yellow', linestyle=':', alpha=0.7,
                label=f"Best: {max(history_cnn['val_acc']):.4f}")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Best validation accuracy: {max(history_cnn['val_acc']):.4f}")

## 5. Grad-CAM Visualization

In [ ]:
from Explainability.gradcam import GradCAM

cnn.eval()
cam_extractor = GradCAM(cnn, target_layer=cnn.layer3)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Grad-CAM: What the CNN Sees in Each Spectrogram', fontsize=14, fontweight='bold')

for col_idx, class_idx in enumerate([0, 1, 2]):
    # Find a sample of this class in test set
    target_class = class_idx + 2  # un-offset
    sample_indices = np.where(y_test == target_class)[0]
    if len(sample_indices) == 0:
        continue
    idx = sample_indices[0]

    X_sample, y_sample = ds_test[idx]
    X_tensor = X_sample.unsqueeze(0)  # (1, 1, H, W)
    spectrogram = X_sample.squeeze().numpy()

    cam = cam_extractor(X_tensor, class_idx=class_idx, device=device)

    axes[0, col_idx].imshow(spectrogram, aspect='auto', origin='lower', cmap='inferno')
    axes[0, col_idx].set_title(f'{CLASS_NAMES[class_idx]}\nSpectrogram', fontsize=11)
    axes[0, col_idx].set_xlabel('Time bins')
    axes[0, col_idx].set_ylabel('Frequency bins')

    axes[1, col_idx].imshow(spectrogram, aspect='auto', origin='lower', cmap='inferno', alpha=0.6)
    axes[1, col_idx].imshow(cam, aspect='auto', origin='lower', cmap='jet', alpha=0.7)
    axes[1, col_idx].set_title(f'Grad-CAM Overlay\n(activation hotspots)', fontsize=11)
    axes[1, col_idx].set_xlabel('Time bins')
    axes[1, col_idx].set_ylabel('Frequency bins')

plt.tight_layout()
plt.savefig('gradcam_visualization.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("🔍 Grad-CAM reveals the CNN focuses on the periodic blade-flash frequency bands.")